# Setup

In [ ]:
"""02_baselines.ipynb — Splits, propensity, S-learner, T-learner.

We start with the plumbing (splits + propensity) and then train the two
simplest meta-learners. The point isn't to crown a winner; it's to
demonstrate the structural failure modes of each, which motivates the
more sophisticated estimators in Phase 3.
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from uplift.data import load_raw
from uplift.splits import make_splits, save_splits, load_splits
from uplift.treatment import (
    FEATURE_COLS,
    get_features,
    make_binary_treatment,
    make_xty,
    naive_ate,
)
from uplift.propensity import (
    PropensityModel,
    estimate_propensity_cv,
    overlap_diagnostics,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

RNG = np.random.default_rng(42)

# Create and save the splits

In [ ]:
df = load_raw()
splits = make_splits(df, seed=42)
save_splits(splits)

for name, sdf in splits.items():
    T = make_binary_treatment(sdf)
    print(f"{name:6s}  n={len(sdf):>6,}  treated_rate={T.mean():.4f}")

# Verify naive ATEs are preserved across splits

In [ ]:
print(f"{'split':6s}  {'visit ATE':>10s}  {'visit SE':>10s}  {'spend ATE':>10s}  {'spend SE':>10s}")
for name, sdf in splits.items():
    v = naive_ate(sdf, "visit")
    s = naive_ate(sdf, "spend")
    print(
        f"{name:6s}  "
        f"{v['ate']:>+10.4f}  {v['ate_se']:>10.4f}  "
        f"{s['ate']:>+10.4f}  {s['ate_se']:>10.4f}"
    )

# Fit the propensity model on train

In [ ]:
train_df = splits["train"]
X_train = get_features(train_df)
T_train = make_binary_treatment(train_df)

propensity = PropensityModel().fit(X_train, T_train)

# In-sample predictions on train (for diagnostic plotting only — don't
# use these for downstream IPS, that requires out-of-fold).
p_train_in = propensity.predict(X_train)
print("In-sample propensity diagnostics:")
for k, v in overlap_diagnostics(p_train_in).items():
    print(f"  {k:25s} {v:.4f}")

# Out-of-fold propensity for downstream use

In [ ]:
p_train_oof = estimate_propensity_cv(X_train, T_train, n_splits=5)
print("Out-of-fold propensity diagnostics:")
for k, v in overlap_diagnostics(p_train_oof).items():
    print(f"  {k:25s} {v:.4f}")

# Save for Phase 4 (IPS, DR-learner)
np.save(PROJECT_ROOT / "data" / "processed" / "propensity_train_oof.npy", p_train_oof)

# Plot the propensity distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(p_train_oof, bins=60, color="#4c78a8", edgecolor="white")
ax.axvline(2 / 3, color="black", linestyle="--", label="true propensity = 2/3")
ax.axvline(0.05, color="red", linestyle=":", alpha=0.6, label="extreme threshold")
ax.axvline(0.95, color="red", linestyle=":", alpha=0.6)
ax.set_xlabel("P(T = 1 | X) — out-of-fold")
ax.set_ylabel("Count")
ax.set_xlim(0, 1)
ax.set_title(
    "Propensity distribution: tightly clustered at 2/3 — randomization confirmed\n"
    "(on observational data this would be wide and pushed against 0 and 1)"
)
ax.legend(loc="upper right")
fig.savefig(FIG_DIR / "06_propensity_distribution.png")
plt.show()

# Cross-check: AUC and Feature Importance

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(T_train, p_train_oof)
print(f"Propensity model AUC: {auc:.4f}")
print(f"  (≈ 0.50 ⇒ treatment is unpredictable ⇒ randomization holds)\n")

importances = pd.Series(
    propensity.model.feature_importances_,
    index=propensity.feature_cols_,
).sort_values(ascending=False)
print("Top 10 'most predictive' features for treatment:")
print(importances.head(10))
print("\nNote: these are noise. Under randomization, NO feature should be")
print("meaningfully predictive of treatment. The model is fitting tiny")
print("residual fluctuations.")

# Imports and train both learners

In [ ]:
from uplift.learners import SLearner, TLearner

# Convenience accessors for the val split
val_df = splits["val"]
X_train_feat = get_features(train_df)
T_train_bin = make_binary_treatment(train_df)
Y_train_visit = train_df["visit"]

X_val_feat = get_features(val_df)
T_val_bin = make_binary_treatment(val_df)
Y_val_visit = val_df["visit"]

print("Training S-learner...")
slearner = SLearner().fit(X_train_feat, T_train_bin, Y_train_visit)
print("Training T-learner...")
tlearner = TLearner().fit(X_train_feat, T_train_bin, Y_train_visit)
print("Done.")

cate_s_val = slearner.predict_cate(X_val_feat)
cate_t_val = tlearner.predict_cate(X_val_feat)

# Compare mean CATE to the naive ATE (sanity floor)

In [ ]:
naive_val = naive_ate(val_df, "visit")["ate"]

print(f"{'Method':<20s}  {'mean CATE':>10s}  {'std CATE':>10s}  {'mean/naive':>12s}")
print("-" * 60)
print(f"{'Naive ATE':<20s}  {naive_val:>+10.4f}  {'':>10s}  {1.0:>12.3f}")
print(
    f"{'S-learner':<20s}  {cate_s_val.mean():>+10.4f}  {cate_s_val.std():>10.4f}  {cate_s_val.mean() / naive_val:>12.3f}"
)
print(
    f"{'T-learner':<20s}  {cate_t_val.mean():>+10.4f}  {cate_t_val.std():>10.4f}  {cate_t_val.mean() / naive_val:>12.3f}"
)

# Visual: CATE distributions side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, cate, name in zip(axes, [cate_s_val, cate_t_val], ["S-learner", "T-learner"]):
    ax.hist(cate, bins=60, color="#4c78a8", edgecolor="white", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(
        naive_val,
        color="#54a24b",
        linestyle="--",
        linewidth=1.4,
        label=f"naive ATE = {naive_val:.3f}",
    )
    ax.axvline(
        cate.mean(),
        color="#e45756",
        linestyle="--",
        linewidth=1.4,
        label=f"mean = {cate.mean():.3f}",
    )
    ax.set_title(f"{name}: predicted CATE on val")
    ax.set_xlabel("Predicted CATE")
    ax.set_ylabel("Count")
    ax.legend(loc="upper right", fontsize=9)

fig.suptitle("S-learner is tightly clustered; T-learner is much more spread out", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_cate_distributions.png")
plt.show()

# Demonstrating S-learner shrinkage with deliberate over-regularization

In [ ]:
# Heavily regularized S-learner — small trees, few of them, large leaves
slearner_strong = SLearner(
    base_params={
        "n_estimators": 50,
        "max_depth": 3,
        "num_leaves": 7,
        "min_child_samples": 500,
    }
).fit(X_train_feat, T_train_bin, Y_train_visit)

cate_s_strong = slearner_strong.predict_cate(X_val_feat)

print("S-learner: effect of regularization on the CATE distribution\n")
print(f"{'Config':<20s}  {'mean CATE':>10s}  {'std CATE':>10s}  {'shrinkage':>12s}")
print("-" * 60)
print(
    f"{'Default (n=200)':<20s}  {cate_s_val.mean():>+10.4f}  {cate_s_val.std():>10.4f}  {cate_s_val.mean() / naive_val:>11.1%}"
)
print(
    f"{'Heavy reg (n=50)':<20s}  {cate_s_strong.mean():>+10.4f}  {cate_s_strong.std():>10.4f}  {cate_s_strong.mean() / naive_val:>11.1%}"
)
print(f"{'Naive ATE':<20s}  {naive_val:>+10.4f}  {'—':>10s}  {1.0:>11.1%}")
print()
print("The heavily regularized model has a smaller mean CATE — the treatment")
print("feature is competing for limited model capacity against other features,")
print("and regularization can effectively zero out its contribution.")
print()
print("This is the structural reason S-learner can be biased toward zero.")

# Demonstrating T-learner variance using per-arm sample sizes

In [ ]:
n_treated = (T_train_bin == 1).sum()
n_control = (T_train_bin == 0).sum()
print(f"Training data per arm:")
print(f"  μ_1 trained on {n_treated:>6,} samples (treated)")
print(f"  μ_0 trained on {n_control:>6,} samples (control)")
print(f"  Ratio: {n_treated / n_control:.2f}x more data for the treated model\n")

# Compare prediction "confidence" by looking at the spread of CATE predictions
# between the two methods. T-learner's wider distribution = it's making
# stronger per-customer claims, some of which are noise.
extreme_cates_s = ((cate_s_val < -0.05) | (cate_s_val > 0.20)).mean()
extreme_cates_t = ((cate_t_val < -0.05) | (cate_t_val > 0.20)).mean()
print(f"Fraction of CATEs outside [-0.05, +0.20]:")
print(f"  S-learner: {extreme_cates_s:.1%}")
print(f"  T-learner: {extreme_cates_t:.1%}")
print()
print("T-learner is much more willing to predict 'this customer has a")
print("strongly negative or strongly positive treatment effect'. Some of")
print("those claims are real (true heterogeneity) and some are noise from")
print("the control-arm model fitting random patterns in less data.")

# Rank agreement between S and T

In [ ]:
from scipy.stats import kendalltau, spearmanr

tau, _ = kendalltau(cate_s_val, cate_t_val)
rho, _ = spearmanr(cate_s_val, cate_t_val)

print(f"Rank agreement between S-learner and T-learner on val set:")
print(f"  Kendall's τ: {tau:.3f}")
print(f"  Spearman ρ:  {rho:.3f}")
print()
if tau < 0.5:
    print("⚠️  Low agreement: the two methods rank customers very differently.")
    print("    Policy choice depends heavily on which method we trust.")
else:
    print("Reasonable agreement on the ranking — both methods see similar")
    print("structure, even if magnitudes differ. The cross-check is encouraging:")
    print("the heterogeneity signal is robust to method choice.")

# Sanity check: top-decile realized lift

In [ ]:
def realized_lift_in_top_k(
    df_eval: pd.DataFrame, cate_pred: np.ndarray, k: float, outcome: str = "visit"
) -> float:
    """Realized treatment effect among the top-k fraction by predicted CATE."""
    n_top = int(len(df_eval) * k)
    top_idx = np.argsort(-cate_pred)[:n_top]
    top = df_eval.iloc[top_idx]
    T_top = make_binary_treatment(top)
    Y_top = top[outcome]
    if T_top.sum() < 10 or (T_top == 0).sum() < 10:
        return np.nan
    return Y_top[T_top == 1].mean() - Y_top[T_top == 0].mean()


print(f"Realized lift on val by decile of predicted CATE (visit outcome):\n")
print(f"{'Method':<12s}  {'top 10%':>10s}  {'top 30%':>10s}  {'bottom 10%':>12s}  {'overall':>10s}")
print("-" * 65)
for name, cate in [("S-learner", cate_s_val), ("T-learner", cate_t_val)]:
    t10 = realized_lift_in_top_k(val_df, cate, 0.10)
    t30 = realized_lift_in_top_k(val_df, cate, 0.30)
    b10 = realized_lift_in_top_k(val_df, -cate, 0.10)  # bottom = top of negated
    print(f"{name:<12s}  {t10:>+10.4f}  {t30:>+10.4f}  {b10:>+12.4f}  {naive_val:>+10.4f}")

# Save CATEs for downstream evaluation

In [ ]:
test_df = splits["test"]
X_test_feat = get_features(test_df)

cate_s_test = slearner.predict_cate(X_test_feat)
cate_t_test = tlearner.predict_cate(X_test_feat)

cate_dir = PROJECT_ROOT / "data" / "processed"
pd.DataFrame(
    {"slearner_visit": cate_s_val, "tlearner_visit": cate_t_val},
    index=val_df.index,
).to_parquet(cate_dir / "cate_val.parquet")

pd.DataFrame(
    {"slearner_visit": cate_s_test, "tlearner_visit": cate_t_test},
    index=test_df.index,
).to_parquet(cate_dir / "cate_test.parquet")

print(f"Saved CATE predictions to {cate_dir / 'cate_*.parquet'}")
print(f"  val:  {len(cate_s_val):,} rows × 2 methods")
print(f"  test: {len(cate_s_test):,} rows × 2 methods")

# Train X-learner

In [ ]:
from uplift.learners import XLearner

print("Training X-learner...")
xlearner = XLearner(propensity_model=propensity).fit(X_train_feat, T_train_bin, Y_train_visit)
print("Done.")

cate_x_val = xlearner.predict_cate(X_val_feat)
cate_x_test = xlearner.predict_cate(X_test_feat)

# Side-by-side comparison of all three meta-learners

In [ ]:
print(f"{'Method':<12s}  {'mean CATE':>10s}  {'std CATE':>10s}  {'mean/naive':>12s}")
print("-" * 52)
print(f"{'Naive ATE':<12s}  {naive_val:>+10.4f}  {'—':>10s}  {1.0:>12.3f}")
print(
    f"{'S-learner':<12s}  {cate_s_val.mean():>+10.4f}  {cate_s_val.std():>10.4f}  {cate_s_val.mean() / naive_val:>12.3f}"
)
print(
    f"{'T-learner':<12s}  {cate_t_val.mean():>+10.4f}  {cate_t_val.std():>10.4f}  {cate_t_val.mean() / naive_val:>12.3f}"
)
print(
    f"{'X-learner':<12s}  {cate_x_val.mean():>+10.4f}  {cate_x_val.std():>10.4f}  {cate_x_val.mean() / naive_val:>12.3f}"
)

# Three-way distribution comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, cate, name in zip(
    axes,
    [cate_s_val, cate_t_val, cate_x_val],
    ["S-learner", "T-learner", "X-learner"],
):
    ax.hist(cate, bins=60, color="#4c78a8", edgecolor="white", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(
        naive_val,
        color="#54a24b",
        linestyle="--",
        linewidth=1.4,
        label=f"naive ATE = {naive_val:.3f}",
    )
    ax.axvline(
        cate.mean(),
        color="#e45756",
        linestyle="--",
        linewidth=1.4,
        label=f"mean = {cate.mean():.3f}",
    )
    ax.set_title(f"{name}: predicted CATE on val")
    ax.set_xlabel("Predicted CATE")
    ax.set_xlim(-0.2, 0.4)  # same x-range across panels for fair comparison
    ax.legend(loc="upper right", fontsize=9)

axes[0].set_ylabel("Count")
fig.suptitle("Distribution of predicted CATEs across three meta-learners", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "08_three_meta_learners.png")
plt.show()

# Rank agreement across all three

In [ ]:
from scipy.stats import kendalltau
import itertools

methods = {
    "S-learner": cate_s_val,
    "T-learner": cate_t_val,
    "X-learner": cate_x_val,
}

print("Pairwise Kendall's τ between methods:")
print(f"{'':12s}  {'S':>8s}  {'T':>8s}  {'X':>8s}")
print("-" * 42)
for name_a, cate_a in methods.items():
    row = f"{name_a:<12s}  "
    for name_b, cate_b in methods.items():
        tau, _ = kendalltau(cate_a, cate_b)
        row += f"{tau:>8.3f}  "
    print(row)

# Update the saved CATE files to include X-learner

In [ ]:
pd.DataFrame(
    {"slearner_visit": cate_s_val, "tlearner_visit": cate_t_val, "xlearner_visit": cate_x_val},
    index=val_df.index,
).to_parquet(PROJECT_ROOT / "data" / "processed" / "cate_val.parquet")

pd.DataFrame(
    {"slearner_visit": cate_s_test, "tlearner_visit": cate_t_test, "xlearner_visit": cate_x_test},
    index=test_df.index,
).to_parquet(PROJECT_ROOT / "data" / "processed" / "cate_test.parquet")

print("Updated cate_val.parquet and cate_test.parquet with X-learner predictions.")

# Realized top-decile lift across all three

In [ ]:
print("Realized lift on val by decile (visit outcome):\n")
print(f"{'Method':<12s}  {'top 10%':>10s}  {'top 30%':>10s}  {'bottom 10%':>12s}  {'overall':>10s}")
print("-" * 65)
for name, cate in methods.items():
    t10 = realized_lift_in_top_k(val_df, cate, 0.10)
    t30 = realized_lift_in_top_k(val_df, cate, 0.30)
    b10 = realized_lift_in_top_k(val_df, -cate, 0.10)
    print(f"{name:<12s}  {t10:>+10.4f}  {t30:>+10.4f}  {b10:>+12.4f}  {naive_val:>+10.4f}")

# Train DR-learner and causal forest

In [ ]:
from uplift.learners import DRLearner
from uplift.forest import CausalForest

print("Training DR-learner (5-fold cross-fitting)...")
drlearner = DRLearner(n_folds=5).fit(X_train_feat, T_train_bin, Y_train_visit)
print(f"  AIPW ATE: {drlearner.aipw_ate_:+.4f}")

print("\nTraining causal forest (500 trees)... this takes 2-5 min.")
cforest = CausalForest(n_estimators=500).fit(X_train_feat, T_train_bin, Y_train_visit)
print("Done.")

cate_d_val = drlearner.predict_cate(X_val_feat)
cate_c_val = cforest.predict_cate(X_val_feat)

cate_d_test = drlearner.predict_cate(X_test_feat)
cate_c_test = cforest.predict_cate(X_test_feat)

# Five-way summary table

In [ ]:
methods = {
    "S-learner": cate_s_val,
    "T-learner": cate_t_val,
    "X-learner": cate_x_val,
    "DR-learner": cate_d_val,
    "CausalForest": cate_c_val,
}

print(f"{'Method':<14s}  {'mean CATE':>10s}  {'std CATE':>10s}  {'mean/naive':>12s}")
print("-" * 54)
print(f"{'Naive ATE':<14s}  {naive_val:>+10.4f}  {'—':>10s}  {1.0:>12.3f}")
for name, cate in methods.items():
    ratio = cate.mean() / naive_val
    print(f"{name:<14s}  {cate.mean():>+10.4f}  {cate.std():>10.4f}  {ratio:>12.3f}")

# Overlay distributions (cleaner than 5 panels)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

palette = {
    "S-learner": "#888",
    "T-learner": "#4c78a8",
    "X-learner": "#54a24b",
    "DR-learner": "#e45756",
    "CausalForest": "#b279a2",
}

for name, cate in methods.items():
    ax.hist(
        cate,
        bins=60,
        color=palette[name],
        alpha=0.45,
        label=f"{name} (μ={cate.mean():.3f}, σ={cate.std():.3f})",
        density=True,
        histtype="stepfilled",
    )

ax.axvline(
    naive_val, color="black", linestyle="--", linewidth=1.4, label=f"naive ATE = {naive_val:.3f}"
)
ax.axvline(0, color="black", linewidth=0.6)
ax.set_xlim(-0.15, 0.30)
ax.set_xlabel("Predicted CATE on val set")
ax.set_ylabel("Density")
ax.set_title("All five CATE distributions on one axis")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "09_all_methods_distributions.png")
plt.show()

# Pairwise Kendall's τ heatmap

In [ ]:
from scipy.stats import kendalltau

names = list(methods.keys())
n_methods = len(names)
tau_matrix = np.zeros((n_methods, n_methods))
for i, n_a in enumerate(names):
    for j, n_b in enumerate(names):
        tau_matrix[i, j], _ = kendalltau(methods[n_a], methods[n_b])

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(tau_matrix, vmin=0, vmax=1, cmap="viridis")
for i in range(n_methods):
    for j in range(n_methods):
        ax.text(
            j,
            i,
            f"{tau_matrix[i, j]:.2f}",
            ha="center",
            va="center",
            color="white" if tau_matrix[i, j] < 0.6 else "black",
        )
ax.set_xticks(range(n_methods))
ax.set_yticks(range(n_methods))
ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_yticklabels(names)
ax.set_title("Pairwise Kendall's τ between methods")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "10_method_agreement_heatmap.png")
plt.show()

# Heterogeneity feature importances from the causal forest

In [ ]:
hetero_importances = cforest.feature_importances().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
hetero_importances.plot.barh(ax=ax, color="#b279a2")
ax.invert_yaxis()
ax.set_xlabel("Heterogeneity importance")
ax.set_title("Causal forest: which features drive treatment-effect heterogeneity")
fig.tight_layout()
fig.savefig(FIG_DIR / "11_heterogeneity_importances.png")
plt.show()

print("\nNote: these are *heterogeneity* importances, not predictive ones.")
print("They answer: 'which features cause the treatment effect to vary across customers?'")
print("Compare to the propensity/outcome predictive importances — they differ.")

# Five-way realized top-decile lift

In [ ]:
print("Realized lift on val (visit outcome):\n")
print(f"{'Method':<14s}  {'top 10%':>10s}  {'top 30%':>10s}  {'bottom 10%':>12s}  {'spread':>10s}")
print("-" * 65)
for name, cate in methods.items():
    t10 = realized_lift_in_top_k(val_df, cate, 0.10)
    t30 = realized_lift_in_top_k(val_df, cate, 0.30)
    b10 = realized_lift_in_top_k(val_df, -cate, 0.10)
    spread = t10 - b10
    print(f"{name:<14s}  {t10:>+10.4f}  {t30:>+10.4f}  {b10:>+12.4f}  {spread:>+10.4f}")

# Save all CATEs

In [ ]:
val_cates = pd.DataFrame(
    {
        "slearner": cate_s_val,
        "tlearner": cate_t_val,
        "xlearner": cate_x_val,
        "drlearner": cate_d_val,
        "causalforest": cate_c_val,
    },
    index=val_df.index,
)

test_cates = pd.DataFrame(
    {
        "slearner": cate_s_test,
        "tlearner": cate_t_test,
        "xlearner": cate_x_test,
        "drlearner": cate_d_test,
        "causalforest": cate_c_test,
    },
    index=test_df.index,
)

val_cates.to_parquet(PROJECT_ROOT / "data" / "processed" / "cate_val.parquet")
test_cates.to_parquet(PROJECT_ROOT / "data" / "processed" / "cate_test.parquet")
print("Saved all 5 methods × {val, test} predictions.")
print(f"  val:  {val_cates.shape}")
print(f"  test: {test_cates.shape}")